In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, get_footprints_from_suite2p)


#from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
#                         correct_drift_single_frame, template_generation, 
 #                        plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0

from utils.calcium import calcium


Operating system:  Windows


Autosaving every 180 seconds


auto.py (22): IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
fname = r'F:\bmi\cohort8\DON-014451\day0\calibration\Image_001_001.raw'


# 
bmi_c = CalibrationTools(fname)
bmi_c.calcium = calcium
#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations

#
bmi_c.std_map = bmi_c.data[:1000].mean(0)

# load suite2p footprints
bmi_c = get_footprints_from_suite2p(bmi_c)

# 
print ("DONE...")

memmap :  (90000, 512, 512)
WARNING ***** Found cells with 0-[Ca] traces :  2
  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (391, 90000)
         self.Fneu (neuropile):  (391, 90000)
         self.iscell (Suite2p cell classifier output):  (930, 2)
              of which number of good cells:  (391,)
         self.spks (deconnvoved spikes):  (391, 90000)
         self.stat (footprints structure):  (391,)
         mean std over all cells :  63.5
# of footprints;  391
DONE...


In [3]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
order_type = 'snr'  # 'snr' or 'f0'

#
bmi_c.compute_roi_traces_f0_and_reorder_cells(order_type)  # this function also coputes the snr / f0s of the cells

# 


computing roi traces for SNR indexing: 100%|███████████████████████████████████████| 9000/9000 [01:19<00:00, 113.92it/s]


In [12]:
#########################################################
########### VISUALIZE CELLS BY SNR OR F0 ##################
#########################################################
#
bmi_c.scale=.001
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 10       # Subsample the time series to go faster;

bmi_c.min_f0 = 500
# visualize traces
bmi_c.vmin=0
bmi_c.vmax=3000
bmi_c.max_n_cells = 50
bmi_c.visualize_traces_snr_order(bmi_c.std_map)


memmap :  (90000, 512, 512)


In [13]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [8,9]
bmi_c.ensemble2 = [20,23]
bmi_c.both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", bmi_c.both)

#
# bmi_c.show_traces_ids(bmi_c.both)
# # ######################################################################
# # ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# # ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles2(bmi_c.std_map)

print ("DONE...")


all cells: [ 8  9 20 23]


100%|███████████████████████████████████████████████████████████████████████████| 90000/90000 [00:11<00:00, 7960.44it/s]


Contour:  [187 353]
cell ids:  [ 8  9 20 23]
DONE...


In [22]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################

# if using binning
bmi_c.binning_flag = True
bmi_c.binning_time = 0.200
bmi_c.smoothing_n_bins = 3

# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 3   # reward lockout in seconds
bmi_c.post_missed_reward_lockout = 10
bmi_c.post_reward_lockout_baseline_min = .3 # want to wait until we drop to 30% of threshold
bmi_c.trial_time = 30
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.25#*0.85

#gr.find_reward_thresholds_low_and_high()
#bmi_c.high = 2
stepper = 0.85
bmi_c.find_reward_thresholds_high_realtime(stepper)  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


####################################
# do not change this
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


nsec recording:  3000 max # of random rewards (i.e. every 30sec)  100
 @0.25% reward rate:  25
 high guess:  4.006032318491682
updated rewards #:  1  for threshold:  4.006032318491682
updated rewards #:  2  for threshold:  3.4051274707179298
updated rewards #:  2  for threshold:  2.8943583501102403
updated rewards #:  8  for threshold:  2.4602045975937044
updated rewards #:  9  for threshold:  2.0911739079546487
updated rewards #:  10  for threshold:  1.7774978217614514
updated rewards #:  17  for threshold:  1.5108731484972338
updated rewards #:  18  for threshold:  1.2842421762226486
updated rewards #:  17  for threshold:  1.0916058497892513
updated rewards #:  17  for threshold:  0.9278649723208636
updated rewards #:  22  for threshold:  0.7886852264727341
updated rewards #:  27  for threshold:  0.670382442501824
updated rewards #:  22  for threshold:  0.7886852264727341
updated rewards #:  22  for threshold:  0.7807983742080068
updated rewards #:  22  for threshold:  0.772990390465

In [23]:
#############################################
########### SAVE DATA #######################
#############################################

#    
text = "day0"
bmi_c = save_calibration_data(bmi_c,text)  

contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
contour is broken, skipping: 
 couldn't save bmi_c.object .... TO FIX!
Done...


npyio.py (713): Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
